# 04_improve_imbalance — Role D: Improvement 2 (Class Imbalance)

**Mục tiêu / giả thuyết:** Target bị mất cân bằng (`Enrolled` là lớp thiểu số, `Graduate` là lớp đa số). Giả thuyết: cân bằng lớp khi train (qua `class_weight` hoặc SMOTE) sẽ tăng recall của `Dropout`/`Enrolled` so với M0, có thể đánh đổi bằng accuracy tổng.

Toàn bộ mô hình dùng **một split chung duy nhất** từ `src.data.get_train_test()`. Chỉ M2b được resample bằng SMOTE, và **chỉ trên tập train sau khi split** — tập test không bao giờ bị chạm vào.

> **Giới hạn cần biết khi đọc notebook này:** các guardrail/assert dưới đây kiểm chứng tính đúng đắn trong **một lần chạy** (no-leakage, cấu hình đúng theo `AGENT.md`). Việc double-run reproducibility, pin version toàn dự án, và audit hash SHA-256 cấp bàn giao (theo kế hoạch hoàn thiện Role D, giai đoạn D0/D1/D6–D10) cần được thực hiện thủ công trên máy chạy thật, không thể xác nhận từ notebook đơn lẻ này.

## 1. Môi trường và import

Cell này ghi lại phiên bản môi trường thực sự dùng để tạo artifact (bắt buộc theo giai đoạn D1 của kế hoạch hoàn thiện Role D), sau đó nhập các helper dùng chung. Notebook không tự cài đặt lại logic load/split dữ liệu hay tính metrics.

In [2]:
%matplotlib inline

import sys
from pathlib import Path

import pandas as pd
from imblearn.over_sampling import SMOTE
from sklearn.tree import DecisionTreeClassifier


def find_repo_root(start: Path | None = None) -> Path:
    """Return the nearest ancestor containing the shared data pipeline."""
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "src" / "data.py").is_file():
            return candidate
    raise FileNotFoundError("Không tìm thấy repo root chứa src/data.py")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.data import get_train_test
from src.evaluate import evaluate_model
from src.visualize import plot_tree_figure

FIGURES_DIR = REPO_ROOT / "figures"
OUTPUTS_DIR = REPO_ROOT / "outputs"
RESULTS_PATH = OUTPUTS_DIR / "results.csv"
CLASS_ORDER = ["Dropout", "Enrolled", "Graduate"]

print(f"Repo root: {REPO_ROOT.name}")

Repo root: Lab2_DecisionTree


In [3]:
import numpy, sklearn, matplotlib, imblearn, scipy

ENV_VERSIONS = {
    "python": sys.version.split()[0],
    "numpy": numpy.__version__,
    "pandas": pd.__version__,
    "scipy": scipy.__version__,
    "scikit-learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "imbalanced-learn": imblearn.__version__,
}
display(pd.Series(ENV_VERSIONS, name="version").to_frame())

# Gate môi trường (Mục 2.3) — dừng trước khi train nếu sai bản canonical
assert sklearn.__version__ == "1.9.0", f"Cần scikit-learn 1.9.0, đang có {sklearn.__version__}"
assert imblearn.__version__ == "0.14.2", f"Cần imbalanced-learn 0.14.2, đang có {imblearn.__version__}"
assert pd.__version__ == "2.3.3", f"Cần pandas 2.3.3, đang có {pd.__version__}"
assert numpy.__version__ == "2.3.4", f"Cần numpy 2.3.4, đang có {numpy.__version__}"
print("Environment gate PASS — đúng bộ version canonical.")

,version
python,3.14.0
numpy,2.3.4
pandas,2.3.3
scipy,1.18.1
scikit-learn,1.9.0
matplotlib,3.11.1
imbalanced-learn,0.14.2


Environment gate PASS — đúng bộ version canonical.


## 2. Lấy split chung một lần

Đây là lần gọi duy nhất tới `get_train_test()`. M2a và M2b dùng cùng tập test để so sánh công bằng với M0. Không có `train_test_split` hay scaler nào khác trong notebook này.

In [4]:
X_train, X_test, y_train, y_test = get_train_test()

assert set(y_train.unique()) == set(CLASS_ORDER)
assert set(y_test.unique()) == set(CLASS_ORDER)

split_summary = pd.DataFrame({
    "train_n": y_train.value_counts().reindex(CLASS_ORDER),
    "test_n": y_test.value_counts().reindex(CLASS_ORDER),
})
display(split_summary)
print(f"X_train: {X_train.shape}; X_test: {X_test.shape}")

,train_n,test_n
Target,,
Dropout,1137,284
Enrolled,635,159
Graduate,1767,442


X_train: (3539, 90); X_test: (885, 90)


## 3. Baseline (M0) cần tham chiếu

Trước khi train M2a/M2b, đọc lại dòng M0 (Role B) từ `results.csv` để có mốc so sánh — không train lại M0, không sửa dòng này.

In [5]:
_m0_ref = pd.read_csv(RESULTS_PATH)
_m0_ref = _m0_ref.loc[(_m0_ref["model_id"] == "M0") & (_m0_ref["author"] == "B")]
assert len(_m0_ref) == 1, "Cần đúng một dòng M0 do Role B tạo trong results.csv"

display(_m0_ref[[
    "model_id", "test_acc", "recall_dropout", "recall_enrolled", "recall_graduate",
    "tree_depth", "n_leaves",
]])

,model_id,test_acc,recall_dropout,recall_enrolled,recall_graduate,tree_depth,n_leaves
0,M0,0.668927,0.679577,0.383648,0.764706,27,634


## 4. M2a — Cân bằng bằng trọng số lớp

`class_weight='balanced'` tăng trọng số lỗi cho các lớp ít mẫu khi xây cây. Dữ liệu đầu vào không bị resample. Model không được tune dựa trên test set.

In [6]:
m2a = DecisionTreeClassifier(class_weight="balanced", random_state=42)
m2a.fit(X_train, y_train)

# Quality gates (giai đoạn D3): audit đúng cấu hình thí nghiệm và support test bất biến.
assert m2a.get_params()["random_state"] == 42
assert m2a.get_params()["class_weight"] == "balanced"
_test_support = y_test.value_counts().reindex(CLASS_ORDER).to_dict()
assert _test_support == {"Dropout": 284, "Enrolled": 159, "Graduate": 442}, (
    f"Test support đã thay đổi so với kỳ vọng: {_test_support}"
)

m2a_result = evaluate_model(
    m2a, X_train, y_train, X_test, y_test,
    model_id="M2a",
    model_name="Decision Tree with Balanced Class Weights",
    params={"class_weight": "balanced", "random_state": 42},
    author="D",
    classification_report_path=OUTPUTS_DIR / "classification_report_M2a.txt",
    confusion_matrix_path=FIGURES_DIR / "D_cm_M2a.png",
)

m2a_tree_path = plot_tree_figure(
    m2a, X_train.columns, m2a.classes_,
    FIGURES_DIR / "D_tree_M2a.png",
    max_depth=4, figsize=(30, 18), dpi=200, fontsize=7,
    title=(
        "M2a Decision Tree — Balanced Class Weights "
        f"(display: through depth 4 of {m2a.get_depth()} total; "
        f"{m2a.get_n_leaves()} total leaves)"
    ),
)

display(pd.DataFrame([m2a_result]))
print(f"Saved tree: {m2a_tree_path.name}")

,model_id,model_name,params,train_acc,test_acc,error_rate,precision_macro,recall_macro,f1_macro,roc_auc_macro,recall_dropout,recall_enrolled,recall_graduate,tree_depth,n_leaves,author
0,M2a,Decision Tree with Balanced Class Weights,"{""class_weight"":""balanced"",""random_state"":42}",1.0,0.650847,0.349153,0.58425,0.587848,0.585409,0.705321,0.679577,0.339623,0.744344,28,696,D


Saved tree: D_tree_M2a.png


In [7]:
m2a_tree_full_path = plot_tree_figure(
    m2a, X_train.columns, m2a.classes_,
    FIGURES_DIR / "D_tree_M2a_full.png",
    max_depth=None, figsize=(60, 30), dpi=150,
    title=(
        "M2a Decision Tree — Balanced Class Weights (full tree, "
        f"depth={m2a.get_depth()}, leaves={m2a.get_n_leaves()})"
    ),
)

print(f"Saved full tree: {m2a_tree_full_path.name}")

Saved full tree: D_tree_M2a_full.png


## 5. M2b — SMOTE chỉ trên tập train

`SMOTE.fit_resample()` được gọi sau khi `get_train_test()` đã chia dữ liệu. Tập test gốc không bị thay đổi, nên không có rò rỉ dữ liệu từ các điểm tổng hợp vào đánh giá. Các guardrail bên dưới xác nhận điều này bằng hash, không chỉ bằng lời khẳng định (giai đoạn D4).

In [8]:
import hashlib
import numpy as np


def _fingerprint(obj) -> tuple:
    """Fingerprint đầy đủ: hash giá trị+index, shape, và cột/dtype (DataFrame) hoặc name/dtype (Series)."""
    values_hash = hashlib.sha256(
        pd.util.hash_pandas_object(obj, index=True).values.tobytes()
    ).hexdigest()
    if isinstance(obj, pd.DataFrame):
        return (values_hash, obj.shape, tuple(obj.columns), tuple(str(dt) for dt in obj.dtypes))
    return (values_hash, obj.shape, obj.name, str(obj.dtype))


X_train_fp_before = _fingerprint(X_train)
X_test_fp_before = _fingerprint(X_test)
y_train_fp_before = _fingerprint(y_train)   # <-- MỚI: trước đây thiếu fingerprint y_train
y_test_fp_before = _fingerprint(y_test)

train_counts_before_smote = y_train.value_counts().reindex(CLASS_ORDER).copy()

smote = SMOTE(sampling_strategy="auto", random_state=42, k_neighbors=5)  # tường minh cả 3 tham số (Mục 5.4)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

train_counts_after_smote = y_train_smote.value_counts().reindex(CLASS_ORDER)
n_synthetic = int(train_counts_after_smote.sum() - train_counts_before_smote.sum())

# --- Guardrails chống leakage (D-02) ---
assert _fingerprint(X_train) == X_train_fp_before, "X_train gốc bị sửa in-place!"
assert _fingerprint(X_test) == X_test_fp_before, "X_test bị thay đổi sau khi gọi SMOTE!"
assert _fingerprint(y_train) == y_train_fp_before, "y_train gốc bị sửa in-place!"
assert _fingerprint(y_test) == y_test_fp_before, "y_test bị thay đổi sau khi gọi SMOTE!"
assert list(X_train_smote.columns) == list(X_train.columns), "Thứ tự cột sau resample bị đổi"
assert not np.isnan(X_train_smote.to_numpy(dtype=float)).any(), "Có NaN sau SMOTE"
assert np.isfinite(X_train_smote.to_numpy(dtype=float)).all(), "Có giá trị Inf sau SMOTE"
assert train_counts_after_smote.nunique() == 1, "sampling_strategy='auto' phải cân bằng tuyệt đối các lớp"
assert set(train_counts_after_smote.tolist()) == {1767}, "Mỗi lớp sau SMOTE phải có đúng 1.767 mẫu"
assert int(train_counts_after_smote.sum()) == 5301, "Tổng train sau SMOTE phải là 5.301"
assert n_synthetic == 1762, f"Số synthetic phải là 1.762, đang là {n_synthetic}"

_test_support_after = y_test.value_counts().reindex(CLASS_ORDER).to_dict()
assert _test_support_after == {"Dropout": 284, "Enrolled": 159, "Graduate": 442}, (
    f"Test support đã thay đổi: {_test_support_after}"
)

smote_summary = pd.DataFrame({
    "before_SMOTE_train": train_counts_before_smote,
    "after_SMOTE_train": train_counts_after_smote,
})
display(smote_summary)
print(f"Resampled train shape: {X_train_smote.shape} (+{n_synthetic} synthetic rows)")
print("Guardrails PASS: X_train/X_test/y_train/y_test gốc bất biến (giá trị+index+shape+cột+dtype).")

,before_SMOTE_train,after_SMOTE_train
Target,,
Dropout,1137,1767
Enrolled,635,1767
Graduate,1767,1767


Resampled train shape: (5301, 90) (+1762 synthetic rows)
Guardrails PASS: X_train/X_test/y_train/y_test gốc bất biến (giá trị+index+shape+cột+dtype).


In [9]:
m2b = DecisionTreeClassifier(random_state=42)
m2b.fit(X_train_smote, y_train_smote)

# Evaluate trên split gốc (X_train, y_train, X_test, y_test), không phải dữ liệu SMOTE —
# tránh train_acc bị "ảo" vì tính trên chính các điểm tổng hợp mô hình đã học thuộc lòng.
m2b_result = evaluate_model(
    m2b, X_train, y_train, X_test, y_test,
    model_id="M2b",
    model_name="Decision Tree with SMOTE on Training Set",
    params={"sampler": "SMOTE", "smote_random_state": 42, "random_state": 42},
    author="D",
    classification_report_path=OUTPUTS_DIR / "classification_report_M2b.txt",
    confusion_matrix_path=FIGURES_DIR / "D_cm_M2b.png",
)

m2b_tree_path = plot_tree_figure(
    m2b, X_train.columns, m2b.classes_,
    FIGURES_DIR / "D_tree_M2b.png",
    max_depth=4, figsize=(30, 18), dpi=200, fontsize=7,
    title=(
        "M2b Decision Tree — Trained on SMOTE-resampled train set "
        f"(display: through depth 4 of {m2b.get_depth()} total; "
        f"{m2b.get_n_leaves()} total leaves)"
    ),
)

display(pd.DataFrame([m2b_result]))
print(f"Saved tree: {m2b_tree_path.name}")

,model_id,model_name,params,train_acc,test_acc,error_rate,precision_macro,recall_macro,f1_macro,roc_auc_macro,recall_dropout,recall_enrolled,recall_graduate,tree_depth,n_leaves,author
0,M2b,Decision Tree with SMOTE on Training Set,"{""random_state"":42,""sampler"":""SMOTE"",""smote_ra...",1.0,0.688136,0.311864,0.636513,0.642937,0.638747,0.742122,0.707746,0.465409,0.755656,39,847,D


Saved tree: D_tree_M2b.png


In [10]:
m2b_tree_full_path = plot_tree_figure(
    m2b, X_train.columns, m2b.classes_,
    FIGURES_DIR / "D_tree_M2b_full.png",
    max_depth=None, figsize=(70, 35), dpi=150,
    title=(
        "M2b Decision Tree — Trained on SMOTE-resampled train set (full tree, "
        f"depth={m2b.get_depth()}, leaves={m2b.get_n_leaves()})"
    ),
)

print(f"Saved full tree: {m2b_tree_full_path.name}")

Saved full tree: D_tree_M2b_full.png


## 6. Audit: hạn chế của vanilla SMOTE trên dữ liệu categorical mã hóa số

Đây là audit định lượng bắt buộc để báo cáo trung thực (giai đoạn D5), **không phải** căn cứ để đổi sang `SMOTENC` — pipeline chung của dự án chỉ one-hot 4 nhóm cột trong `src/data.py`; các cột categorical cardinality cao khác (ví dụ Mother's/Father's qualification, occupation, Nacionality — xem `docs/feature_types.md`) vẫn giữ dạng mã số nguyên. SMOTE gốc coi mọi feature là liên tục, nên có thể nội suy ra giá trị mã categorical không tồn tại. Ô dưới đây đo hiện tượng này thay vì chỉ mô tả chung chung.

> Danh sách cột bên dưới cần được đối chiếu lại với `docs/feature_types.md` trước khi đưa số liệu vào báo cáo — notebook chỉ lọc theo tên cột có sẵn trong `X_train`, không tự suy luận ý nghĩa nghiệp vụ.

In [11]:
# D-03: prefix-match thay vì tuple membership (tuple membership có thể bỏ sót synthetic
# trùng hàng gốc, vì hai hàng khác nhau nhưng giống hệt giá trị sẽ bị coi nhầm là "gốc")
X_train_reset = X_train.reset_index(drop=True)
X_train_smote_reset = X_train_smote.reset_index(drop=True)
n_original = len(X_train_reset)

prefix_matches = X_train_smote_reset.iloc[:n_original].equals(X_train_reset)
assert prefix_matches, (
    "Phần đầu output resampled không khớp bit-for-bit X_train gốc theo đúng thứ tự — "
    "dừng lại, không tự suy đoán vị trí synthetic rows."
)

synthetic_rows = X_train_smote_reset.iloc[n_original:]
assert len(synthetic_rows) == n_synthetic == 1762
print(f"Xác nhận bằng prefix-match: {n_original} hàng đầu khớp bit-for-bit X_train gốc; "
      f"{len(synthetic_rows)} hàng synthetic.")

Xác nhận bằng prefix-match: 3539 hàng đầu khớp bit-for-bit X_train gốc; 1762 hàng synthetic.


In [12]:
# D-04/D-05: audit đầy đủ — binary + tổng=1 + đúng 1 category active, KHÔNG chỉ kiểm tra tổng
CATEGORICAL_CODE_COLUMNS_NOMINAL = [c for c in [
    "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation", "Nacionality",
] if c in X_train.columns]
CATEGORICAL_CODE_COLUMNS_ORDINAL = [c for c in ["Application order"] if c in X_train.columns]
CATEGORICAL_CODE_COLUMNS = CATEGORICAL_CODE_COLUMNS_NOMINAL + CATEGORICAL_CODE_COLUMNS_ORDINAL

code_audit_rows = []
for col in CATEGORICAL_CODE_COLUMNS:
    observed_codes = set(X_train[col].unique().tolist())
    vals = synthetic_rows[col].to_numpy(dtype=float)
    code_audit_rows.append({
        "column": col,
        "type": "nominal" if col in CATEGORICAL_CODE_COLUMNS_NOMINAL else "ordinal",
        "frac_non_integer": float(np.mean(~np.isclose(vals, np.round(vals)))),
        "frac_out_of_train_vocab": float(np.mean([v not in observed_codes for v in np.round(vals)])),
        "n_unique_before": int(X_train[col].nunique()),
    })
code_audit = pd.DataFrame(code_audit_rows)
display(code_audit)

_binary_cols = [c for c in X_train.columns if set(X_train[c].unique()).issubset({0, 1})]
_onehot_groups = {}
for c in _binary_cols:
    _onehot_groups.setdefault(c.split("_")[0], []).append(c)
_onehot_groups = {k: v for k, v in _onehot_groups.items() if len(v) > 1}

onehot_audit_rows = []
for prefix, cols in _onehot_groups.items():
    block = synthetic_rows[cols].to_numpy(dtype=float)
    row_sums = block.sum(axis=1)
    is_binary = np.isclose(block, 0.0) | np.isclose(block, 1.0)
    all_binary = is_binary.all(axis=1)
    active_count = np.isclose(block, 1.0).sum(axis=1)
    valid_one_hot = all_binary & np.isclose(row_sums, 1.0) & (active_count == 1)
    onehot_audit_rows.append({
        "group": prefix,
        "n_synthetic": len(block),
        "fractional_component_rate": float(np.mean(~all_binary)),
        "sum_zero_rate": float(np.mean(np.isclose(row_sums, 0.0))),
        "sum_one_rate": float(np.mean(np.isclose(row_sums, 1.0))),
        "sum_greater_than_one_rate": float(np.mean(row_sums > 1.0 + 1e-9)),
        "active_count_violation_rate": float(np.mean(active_count != 1)),
        "valid_one_hot_rate": float(np.mean(valid_one_hot)),
    })
onehot_audit = pd.DataFrame(onehot_audit_rows)
display(onehot_audit)
print(
    "Ghi chú cơ chế: imbalanced-learn 0.14.2 khôi phục dtype DataFrame đầu ra bằng "
    "ArraysTransformer.astype(dtypes gốc) — https://github.com/scikit-learn-contrib/"
    "imbalanced-learn/blob/0.14.2/imblearn/utils/_validation.py . Vì cột one-hot có dtype "
    "int, giá trị nội suy phân số bị cắt cụt về int, nên vi phạm đo được nhiều khả năng là "
    "sum_zero_rate (all-zero), không phải sum_greater_than_one_rate (đa category)."
)

,column,type,frac_non_integer,frac_out_of_train_vocab,n_unique_before
0,Mother's qualification,nominal,0.0,0.030079,28
1,Father's qualification,nominal,0.0,0.028944,32
2,Mother's occupation,nominal,0.0,0.011918,31
3,Father's occupation,nominal,0.0,0.020431,41
4,Nacionality,nominal,0.0,0.023837,19
5,Application order,ordinal,0.0,0.000000,8


,group,n_synthetic,fractional_component_rate,sum_zero_rate,sum_one_rate,sum_greater_than_one_rate,active_count_violation_rate,valid_one_hot_rate
0,Marital Status,1762,0.0,0.146992,0.853008,0.0,0.146992,0.853008
1,Application mode,1762,0.0,0.653802,0.346198,0.0,0.653802,0.346198
2,Course,1762,0.0,0.830306,0.169694,0.0,0.830306,0.169694
3,Previous qualification,1762,0.0,0.248014,0.751986,0.0,0.248014,0.751986


Ghi chú cơ chế: imbalanced-learn 0.14.2 khôi phục dtype DataFrame đầu ra bằng ArraysTransformer.astype(dtypes gốc) — https://github.com/scikit-learn-contrib/imbalanced-learn/blob/0.14.2/imblearn/utils/_validation.py . Vì cột one-hot có dtype int, giá trị nội suy phân số bị cắt cụt về int, nên vi phạm đo được nhiều khả năng là sum_zero_rate (all-zero), không phải sum_greater_than_one_rate (đa category).


## 7. So sánh recall từng lớp: M0, M2a, M2b

Đây là tiêu chí chính của cải tiến xử lý mất cân bằng. Nếu test accuracy tổng của M2a hoặc M2b giảm so với M0, đó là kết quả bình thường và có thể được mong đợi: mô hình đang đánh đổi một phần độ chính xác trên lớp đa số để nhận diện tốt hơn các lớp thiểu số, đặc biệt `Enrolled`. Không tối ưu lại chỉ nhằm đưa accuracy trở về bằng M0; cần đánh giá đánh đổi bằng recall của từng lớp. Bảng dưới đây được tính lại bằng code từ `results.csv`, không gõ tay.

In [13]:
results = pd.read_csv(RESULTS_PATH)
required_rows = {"M0", "M2a", "M2b"}
assert required_rows.issubset(set(results["model_id"])), "Thiếu kết quả M0/M2a/M2b trong results.csv"

m0_rows = results.loc[(results["model_id"] == "M0") & (results["author"] == "B")]
assert len(m0_rows) == 1, "Cần đúng một dòng M0 do Role B tạo"

comparison = (
    results.loc[results["model_id"].isin(["M0", "M2a", "M2b"])]
    .set_index("model_id")
    .loc[["M0", "M2a", "M2b"], [
        "model_name", "test_acc", "recall_dropout",
        "recall_enrolled", "recall_graduate",
    ]]
    .rename(columns={
        "test_acc": "test_accuracy",
        "recall_dropout": "recall_Dropout",
        "recall_enrolled": "recall_Enrolled",
        "recall_graduate": "recall_Graduate",
    })
)

recall_deltas_vs_m0 = comparison.loc[["M2a", "M2b"], [
    "recall_Dropout", "recall_Enrolled", "recall_Graduate"
]].subtract(comparison.loc["M0", [
    "recall_Dropout", "recall_Enrolled", "recall_Graduate"
]], axis=1)

display(comparison.style.format({
    "test_accuracy": "{:.4f}",
    "recall_Dropout": "{:.4f}",
    "recall_Enrolled": "{:.4f}",
    "recall_Graduate": "{:.4f}",
}))
display(recall_deltas_vs_m0.style.format("{:+.4f}").set_caption("Recall change versus M0"))

,model_name,test_accuracy,recall_Dropout,recall_Enrolled,recall_Graduate
model_id,,,,,
M0,Baseline Decision Tree (Gini),0.6689,0.6796,0.3836,0.7647
M2a,Decision Tree with Balanced Class Weights,0.6508,0.6796,0.3396,0.7443
M2b,Decision Tree with SMOTE on Training Set,0.6881,0.7077,0.4654,0.7557


,recall_Dropout,recall_Enrolled,recall_Graduate
model_id,,,
M2a,+0.0000,-0.0440,-0.0204
M2b,+0.0282,+0.0818,-0.0090


## 8. Kết luận và quality gates

Dùng bảng recall phía trên cùng các confusion matrix (`D_cm_M2a.png`, `D_cm_M2b.png`) và hình cây (`D_tree_M2a.png`, `D_tree_M2a_full.png`, `D_tree_M2b.png`) để viết phần báo cáo f.2. Kết luận phải nêu đồng thời các cải thiện/giảm sút recall theo từng lớp và thay đổi accuracy tổng, thay vì chỉ chọn model có accuracy cao nhất.

**Đối soát số học tối thiểu (một phần giai đoạn D7)** — chạy cell dưới để tự kiểm tra tính nhất quán nội bộ của hai dòng M2a/M2b trước khi coi notebook là hoàn tất.

In [14]:
for _mid, _res in (("M2a", m2a_result), ("M2b", m2b_result)):
    assert abs(_res["error_rate"] - (1 - _res["test_acc"])) < 1e-9, f"{_mid}: error_rate != 1 - test_acc"
    for _k in ("precision_macro", "recall_macro", "f1_macro", "roc_auc_macro"):
        assert _res[_k] is not None and 0.0 <= _res[_k] <= 1.0, f"{_mid}: {_k} ngoài [0, 1] hoặc null"

assert (pd.read_csv(RESULTS_PATH)["model_id"] == "M0").sum() == 1
assert (pd.read_csv(RESULTS_PATH)["model_id"] == "M1").sum() == 1
assert (pd.read_csv(RESULTS_PATH)["model_id"] == "M2a").sum() == 1
assert (pd.read_csv(RESULTS_PATH)["model_id"] == "M2b").sum() == 1
assert (pd.read_csv(RESULTS_PATH)["model_id"] == "M3").sum() == 1
assert pd.read_csv(RESULTS_PATH).loc[pd.read_csv(RESULTS_PATH)["model_id"].isin(["M2a", "M2b"]), "author"].eq("D").all()

print("Quality gate nội bộ: PASS (error_rate khớp test_acc; macro metrics trong [0, 1]).")
print(
    "Lưu ý: đây chỉ là đối soát trong 1 lần chạy. Việc chạy lặp lần 2 trên kernel sạch để "
    "xác nhận kết quả giống hệt (giai đoạn D7 của kế hoạch) cần thực hiện thủ công."
)

Quality gate nội bộ: PASS (error_rate khớp test_acc; macro metrics trong [0, 1]).
Lưu ý: đây chỉ là đối soát trong 1 lần chạy. Việc chạy lặp lần 2 trên kernel sạch để xác nhận kết quả giống hệt (giai đoạn D7 của kế hoạch) cần thực hiện thủ công.
